# Fake News Classifier
**Model:** Logistic Regression + TF-IDF (bigrams)
**Dataset:** WELFake — 72,134 articles from Kaggle, 63,557 after removing exact-duplicate rows
**Test Accuracy (deduped, main model):** 95.22%

## Label mapping
`0` = Real, `1` = Fake. Verified by inspecting sampled titles per label — do not assume label 1 is Real.

## Findings
- **Leakage:** the original 80/20 split (before dedup) let 2,656 of 14,427 test rows (18.4%) duplicate a training row. Fitting on that split gives 95.76% accuracy, but that number is inflated by memorized duplicates. Deduplicating on the cleaned text before splitting gives the honest number: **95.22%**.
- **Title-only baseline:** 88.91% (full title+text combined model still beats it by ~6.3 points).
- **Naive Bayes underperformed LR:** 86.27% vs 95.22% on the same combined features.
- **Source/format leakage in features:** removing a fixed list of source/format tokens (`reuters`, `via`, `image`, weekday/month names, etc.) before vectorizing drops accuracy from 95.22% to 93.54% — roughly 1.7 points of the model's accuracy comes from recognizing wire-service and formatting artifacts, not article content. See the ablation cells below for the full token lists before and after.

In [1]:
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /Users/jaden-
[nltk_data]     isaac/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
df = pd.read_csv('/Users/jaden-isaac/Documents/GitHub/news-headline-classifier/dataset/WELFake_Dataset.csv')
print(df.shape)
print(df.head())
print(df['label'].value_counts())  # 0 = Real, 1 = Fake

(72134, 4)
   Unnamed: 0                                              title  \
0           0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
1           1                                                NaN   
2           2  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...   
3           3  Bobby Jindal, raised Hindu, uses story of Chri...   
4           4  SATAN 2: Russia unvelis an image of its terrif...   

                                                text  label  
0  No comment is expected from Barack Obama Membe...      1  
1     Did they post their votes for Hillary already?      1  
2   Now, most of the demonstrators gathered last ...      1  
3  A dozen politically active pastors came here f...      0  
4  The RS-28 Sarmat missile, dubbed Satan 2, will...      1  
label
1    37106
0    35028
Name: count, dtype: int64


In [3]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

df['combined'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df['clean_combined'] = df['combined'].apply(clean_text)
print(df['clean_combined'].head())

0    law enforcement high alert following threats c...
1                           post votes hillary already
2    unbelievable obamas attorney general says char...
3    bobby jindal raised hindu uses story christian...
4    satan russia unvelis image terrifying new supe...
Name: clean_combined, dtype: object


In [4]:
# Leakage check: how many test rows would duplicate a train row under the old (non-deduped) split
X_leaky = df['clean_combined']
y_leaky = df['label']
X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky, y_leaky, test_size=0.2, random_state=42
)

train_leaky_set = set(X_train_leaky)
leaked_test_rows = X_test_leaky.isin(train_leaky_set).sum()
print(f"Rows before dedupe: {len(df)}")
print(f"Test rows with an exact duplicate in train (old split): {leaked_test_rows} / {len(X_test_leaky)}")

vectorizer_leaky = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_leaky_tfidf = vectorizer_leaky.fit_transform(X_train_leaky)
X_test_leaky_tfidf = vectorizer_leaky.transform(X_test_leaky)
model_leaky = LogisticRegression(max_iter=1000)
model_leaky.fit(X_train_leaky_tfidf, y_train_leaky)
leaky_accuracy = accuracy_score(y_test_leaky, model_leaky.predict(X_test_leaky_tfidf))
print(f"Old (leaky) split accuracy: {leaky_accuracy:.4f}")

# Deduplicate on clean_combined before splitting, so the leak above can't recur
df = df.drop_duplicates(subset='clean_combined', keep='first').reset_index(drop=True)
print(f"Rows after dedupe: {len(df)}")

Rows before dedupe: 72134
Test rows with an exact duplicate in train (old split): 2656 / 14427


Old (leaky) split accuracy: 0.9576


Rows after dedupe: 63557


In [5]:
X = df['clean_combined']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [6]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [7]:
y_pred = model.predict(X_test_tfidf)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))

Accuracy: 0.9522
              precision    recall  f1-score   support

        Real       0.95      0.96      0.96      6873
        Fake       0.95      0.94      0.95      5839

    accuracy                           0.95     12712
   macro avg       0.95      0.95      0.95     12712
weighted avg       0.95      0.95      0.95     12712



In [8]:
print(df.columns.tolist())

['Unnamed: 0', 'title', 'text', 'label', 'combined', 'clean_combined']


In [9]:
# Title-only baseline (same deduped dataset)
df['clean_title'] = df['title'].fillna('').apply(clean_text)

Xt = df['clean_title']
yt = df['label']
Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    Xt, yt, test_size=0.2, random_state=42
)
vec_title = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
Xt_train_tfidf = vec_title.fit_transform(Xt_train)
Xt_test_tfidf = vec_title.transform(Xt_test)

model_title = LogisticRegression(max_iter=1000)
model_title.fit(Xt_train_tfidf, yt_train)
title_only_acc = accuracy_score(yt_test, model_title.predict(Xt_test_tfidf))
print(f"Title-only accuracy: {title_only_acc:.4f}")

Title-only accuracy: 0.8891


In [10]:
# Naive Bayes comparison, same combined features as the main model
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_acc = accuracy_score(y_test, nb_model.predict(X_test_tfidf))
print(f"MultinomialNB accuracy (combined features): {nb_acc:.4f}")

MultinomialNB accuracy (combined features): 0.8627


In [11]:
# What the model learned: top 20 tokens per class (label 0 = Real, label 1 = Fake)
feature_names = vectorizer.get_feature_names_out()
coefs = model.coef_[0]

top_fake_idx = coefs.argsort()[-20:][::-1]
top_real_idx = coefs.argsort()[:20]

print("Top 20 tokens pushing toward FAKE (label 1):")
for i in top_fake_idx:
    print(f"  {feature_names[i]:<25} {coefs[i]:.4f}")

print("\nTop 20 tokens pushing toward REAL (label 0):")
for i in top_real_idx:
    print(f"  {feature_names[i]:<25} {coefs[i]:.4f}")

Top 20 tokens pushing toward FAKE (label 1):
  via                       13.0161
  video                     8.5618
  october                   8.3052
  image                     8.1423
  image via                 7.2488
  november                  6.6507
  hillary                   6.3925
  images                    4.5398
  breaking                  4.3773
  trump                     4.2923
  obama                     4.2491
  fbi                       3.7381
  america                   3.6425
  however                   3.4788
  wire                      3.4700
  president trump           3.3104
  share                     3.3047
  today                     3.2799
  dc                        3.1811
  please                    3.1270

Top 20 tokens pushing toward REAL (label 0):
  reuters                   -21.4511
  said                      -15.0738
  breitbart                 -12.9487
  trumps                    -8.2619
  washington reuters        -8.2510
  twitter                

In [12]:
# Ablation: strip source/format tokens before vectorizing, retrain, compare accuracy
ablation_tokens = {
    'reuters', 'washington', 'york', 'times', 'via', 'image', 'images', 'video',
    'twitter', 'pic', 'com', 'rt', 'featured', 'getty', 'wire', 'breaking', 'share', 'source', 'follow',
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
    'january', 'february', 'march', 'april', 'may', 'june', 'july', 'august',
    'september', 'october', 'november', 'december'
}
print(f"Ablated tokens ({len(ablation_tokens)}):", sorted(ablation_tokens))

def remove_ablation_tokens(text):
    return ' '.join(t for t in text.split() if t not in ablation_tokens)

df['clean_combined_ablated'] = df['clean_combined'].apply(remove_ablation_tokens)

X_train_abl = df.loc[X_train.index, 'clean_combined_ablated']
X_test_abl = df.loc[X_test.index, 'clean_combined_ablated']

vectorizer_abl = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_train_abl_tfidf = vectorizer_abl.fit_transform(X_train_abl)
X_test_abl_tfidf = vectorizer_abl.transform(X_test_abl)

model_abl = LogisticRegression(max_iter=1000)
model_abl.fit(X_train_abl_tfidf, y_train)
ablation_acc = accuracy_score(y_test, model_abl.predict(X_test_abl_tfidf))

print(f"Accuracy WITH source/format tokens:    {accuracy_score(y_test, y_pred):.4f}")
print(f"Accuracy WITHOUT source/format tokens: {ablation_acc:.4f}")

Ablated tokens (38): ['april', 'august', 'breaking', 'com', 'december', 'featured', 'february', 'follow', 'friday', 'getty', 'image', 'images', 'january', 'july', 'june', 'march', 'may', 'monday', 'november', 'october', 'pic', 'reuters', 'rt', 'saturday', 'september', 'share', 'source', 'sunday', 'thursday', 'times', 'tuesday', 'twitter', 'via', 'video', 'washington', 'wednesday', 'wire', 'york']


Accuracy WITH source/format tokens:    0.9522
Accuracy WITHOUT source/format tokens: 0.9354


In [13]:
# Top tokens after ablation
feature_names_abl = vectorizer_abl.get_feature_names_out()
coefs_abl = model_abl.coef_[0]

top_fake_idx_abl = coefs_abl.argsort()[-20:][::-1]
top_real_idx_abl = coefs_abl.argsort()[:20]

print("After ablation - top 20 tokens pushing toward FAKE (label 1):")
for i in top_fake_idx_abl:
    print(f"  {feature_names_abl[i]:<25} {coefs_abl[i]:.4f}")

print("\nAfter ablation - top 20 tokens pushing toward REAL (label 0):")
for i in top_real_idx_abl:
    print(f"  {feature_names_abl[i]:<25} {coefs_abl[i]:.4f}")

After ablation - top 20 tokens pushing toward FAKE (label 1):
  hillary                   6.9880
  obama                     4.9719
  trump                     4.8879
  however                   4.5409
  america                   4.0720
  watch                     3.9809
  entire                    3.7875
  today                     3.6959
  president trump           3.6013
  fbi                       3.5985
  even                      3.5156
  please                    3.4994
  dc                        3.4614
  yearold                   3.4143
  flickr                    3.4070
  article                   3.3751
  fact                      3.3313
  mosul                     3.2988
  know                      3.2100
  photo                     3.1319

After ablation - top 20 tokens pushing toward REAL (label 0):
  said                      -18.1516
  breitbart                 -15.2081
  president donald          -11.0098
  trumps                    -8.4373
  https                     